# Obsługa ROS Service

## Używanie komend z terminala

### Wprowadzenie

ROS Service jest kolejnym sposobem komunikacji. Zaletą serwisów jest możliwość wysłania zapytania do serwera. Podobnie jak dla ROS topic należy znać format wiadomości. Serwisy w danej paczce ROS przechowywane są w katalogu srv, a rozszerzenie serwisu to **.srv**. 

#### Struktura wiadomości serwisowej

Wyróżnia się podział na format wiadomości:
- wysyłany przez klienta (górna część wiadomości nad znakiem **---**)
- odpowiedź serwera (dolna część wiadomości pod znakiem **---**)

Z lewej strony należy podać typ wiadomości ROS. Mogą być one bardziej złożone i składać się z już utworzonych
wiadomości (ROS msg). Z prawej strony podawana jest nazwa pola.

In [3]:
# Dla obsługi wiadomości serwisowych używane jest polecenie rossrv. Polecenie można wywołać
# z następującymi parametrami:
!rossrv --help

rossrv is a command-line tool for displaying information about ROS Service types.

Commands:
	rossrv show	Show service description
	rossrv info	Alias for rossrv show
	rossrv list	List all services
	rossrv md5	Display service md5sum
	rossrv package	List services in a package
	rossrv packages	List packages that contain services

Type rossrv <command> -h for more detailed usage



Typ serwisu to **nazwa paczki + nazwa serwisu.srv**. Wyświetlenie przykładowego serwisu znajdującego się w paczce tsr_materials:

In [1]:
!rossrv show pkg_tsr/Task

float64 x
float64 y
---
string result



In [4]:
!rossrv info pkg_tsr/Task

float64 x
float64 y
---
string result



#### Wyświetlanie dostępnej pomocy dla polecenia rosservice.

In [7]:
!rosservice --help

Commands:
	rosservice args	print service arguments
	rosservice call	call the service with the provided args
	rosservice find	find services by service type
	rosservice info	print information about service
	rosservice list	list active services
	rosservice type	print service type
	rosservice uri	print service ROSRPC uri

Type rosservice <command> -h for more detailed usage, e.g. 'rosservice call -h'



#### Wyświetlanie listy aktualnie dostępnych serwisów

In [6]:
# Dostępne serwisy wraz z uruchomioną symulacją turtlesim
!rosservice list

/clear
/kill
/reset
/rosout/get_loggers
/rosout/set_logger_level
/spawn
/turtle1/set_pen
/turtle1/teleport_absolute
/turtle1/teleport_relative
/turtlesim/get_loggers
/turtlesim/set_logger_level


### Dostępne serwisy dla turtlesim node

W podstawowym zakresie dostępne są następujące serwisy bez względu na robota:
- /clear - wyczyszczenie narysowanych ścieżek
- /kill - usunięcie robota
- /reset - reset środowiska do stanu początkowego
- /spawn - utworzenie nowego robota

Dla pojedynczego utworzonego robota w przestrzeni nazw na przykładzie **turtle1** dostępne są nastpujące serwisy:
- /turtle1/set_pen - ustawienie koloru pędzla do rysowania
- /turtle1/teleport_absolute - natychmiastowe przeniesienie robota do wskazanej lokalizacji
- /turtle1/teleport_relative - przeniesienie robota, współrzędne podawane w układzie robota

In [8]:
# Dostępne parametry dla serwisu tworzącego nowego żółwia
!rosservice args /spawn

x y theta name


Wyświetlenie informacji o serwisie "clear". Informuje o nazwie noda z którego pochodzi serwis, typie wiadomości i przyjmowanych argumentach.

In [ ]:
!rosservice info clear

### Wywołanie serwisu

Do wywołania serwisu używamy rosservice call a następnie w kolejności podajemy argumenty

*rosservice call argument1 argument2 ...*

**Uwaga! Współrzędne x,y powinny być z przedziału <0,11>**

Tworzenie nowego robota.

In [ ]:
!rosservice call spawn 15 15 0 t5

Z poziomu terminala można wpisać "rosservice call spawn" i kliknąć kilka razy tabulator, a wiadomość wraz z argumentami powinna uzupełnić się domyślnymi parametrami.

rosservice call spawn "x: 0.0

y: 0.0

theta: 0.0

name: ''" 

### Sprawdzenie typu wiadomości serwisowej

In [35]:
!rosservice type spawn

turtlesim/Spawn


## Używanie serwisów w Pythonie - klient

In [37]:
import rospy
rospy.init_node("serwis_node_test")

Serwis */clear* do czyszczenia mapy, bez argumentów. Po stronie klienta do obsługi serwisu używamy *ServiceProxy* z biblioteki *rospy*. Jako pierwszy argument podawana jest nazwa serwisu z którego ma być odebrana odpowiedź, a jako drugi argument podawany jest typ serwisu.

In [38]:
#import typu wiadomości serwisowej
from std_srvs.srv import Empty

In [40]:
# Utworzenie klienta "ServiceProxy"
clear_map = rospy.ServiceProxy('clear', Empty)
# Wysłanie zapytania przez klienta. Dla pustego zapytania argument nie jest 
# przekazywany do funkcji
clear_map()

# To samo wysłanie zapytania przez klienta tylko z utworzeniem pustego zapytania
# Następuje dodatkowo dodanie do importu EmptyRequest <- zapytanie wysyłane przez klienta
from std_srvs.srv import EmptyRequest
request = EmptyRequest()
clear_map(request)

In [42]:
# Czysczenie przestrzeni z robotów */reset*
reset_sim_state = rospy.ServiceProxy('reset', Empty)
reset_sim_state()

Używanie serwisu z argumentami na przykładzie */spawn*.  Kolejne argumenty polecenia podajemy po przecinku.

In [ ]:
from turtlesim.srv import Spawn

In [ ]:
create_new_robot = rospy.ServiceProxy('spawn', Spawn)
# Dla serwisu Spawn konieczne jest podanie kolejnych argmentów. Można je podać jako kolejne
# argumenty w trakcie wyslania zapytania
create_new_robot(3, 3, 0, "t8")

# lub można utworzyć wiadomość z zapytaniem
from turtlesim.srv import SpawnRequest
request = SpawnRequest()
request.x 7
request.y = 10
request.theta = 3.14
request.name = "nowy2"
create_new_robot(request)

In [ ]:
# Zamiast oddzielny importów Spawn i SpawnRequest można zapisać
from turtlesim.srv import Spawn, SpawnRequest
# lub można zaimportować wszystkie wiadomości serwisowe z paczki przy użyciu *
from turtlesim.srv import *

## Używanie serwisów w Pythonie - serwer

In [ ]:
# czyszczenie symulacji
!rosservice call reset

In [45]:
from pkg_tsr.srv import Task
from turtlesim.srv import TeleportAbsolute
from std_srvs.srv import Empty

In [46]:
# Zapis I
def draw_square_function(req):
    # req - przekazany argument, dane wysłane przez klienta, zapytanie typu TaskRequest
    width = 5
    print("Init pose x=%s, y=%s" % (req.x, req.y))
    # utworzenie niezbędnych klientów do teleportacji żółwia i narysowania kwadratów
    set_pose = rospy.ServiceProxy('/turtle1/teleport_absolute', TeleportAbsolute)
    clear_map = rospy.ServiceProxy('clear', Empty)
    set_pose(req.x,req.y,0)
    clear_map()
    set_pose(req.x,req.y + width,0)
    set_pose(req.x + width,req.y + width,0)
    set_pose(req.x + width, req.y,0)
    set_pose(req.x, req.y,0)
    # dla wiadomości serwisowej typu Task odpowiedź (response) jest typu string, dlatego w 
    # return zapisano od razu wartość tekstową
    return "finished"

In [ ]:
# Zapis II (analogiczny do powyższego tylko tworzona jest wiadomośc z odpowiedzią
def draw_square_function(req):
    # req - przekazany argument, dane wysłane przez klienta, zapytanie typu TaskRequest
    width = 5
    print("Init pose x=%s, y=%s" % (req.x, req.y))
    # utworzenie niezbędnych klientów do teleportacji żółwia i narysowania kwadratów
    set_pose = rospy.ServiceProxy('/turtle1/teleport_absolute', TeleportAbsolute)
    clear_map = rospy.ServiceProxy('clear', Empty)
    set_pose(req.x,req.y,0)
    clear_map()
    set_pose(req.x,req.y + width,0)
    set_pose(req.x + width,req.y + width,0)
    set_pose(req.x + width, req.y,0)
    set_pose(req.x, req.y,0)
    # dla wiadomości serwisowej typu Task odpowiedź (response) jest typu string, dlatego w 
    # return zapisano od razu wartość tekstową
    response = TaskResponse()
    response.result = "finished"
    return response

Po stronie serwera używamy obiektu *Service* z biblioteki *rospy*. W kolejności podajemy 3 następujące argumenty:

    - nazwa serwisu
    - typ serwisu
    - nazwa funkcji, która ma zostać wywołan po pojawieniu się żądania od klienta na tym serwisie

In [47]:
# obiekt serwera. Po wysłaniu zapytania (typTask) od klienta na serwer o nazwie 
# robot_teleport wywołana zostanie funkcja draw_square_function
s = rospy.Service('robot_teleport', Task, draw_square_function)

#### UWAGA
Wyłączenie serwisu - gdy wprowadzone zostaną jakieś zmiany w funkcji serwisowej, aby nie restartować Kernela Jupyter Notebook, można poniższą metodą shutdown() zatrzymać działający serwis.

In [ ]:
s.shutdown()

#### Wynik działania powyższego serwisu po wywołaniu klienta

In [48]:
draw_square = rospy.ServiceProxy('robot_teleport', Task)
resp = draw_square(3, 2)

Init pose x=3.0, y=2.0


Odczyt otrzymanej wartości z serwera

In [49]:
resp.result

'finished'

## Dodatek

Tworzenie własnych wiadomości serwisowych. W paczce znajduje się katalog srv w którym można utworzyć dodatkowe wiadomości serwisowe. Konfiguracja w przedstawionym dodatku ogranicza się do używania typów wiadomości z poniższego linku:

http://docs.ros.org/en/melodic/api/std_msgs/html/index-msg.html

In [ ]:
!rossrv show pkg_tsr/Task

In [ ]:
!rosmsg show std_msgs/ColorRGBA

Przykładowo rozbudowany serwis o wiadomość typu ColorRGBA może wyglądać następująco. 

In [ ]:
float64 x
float64 y
ColorRGBA my_color
---
string result

Dodanie nowej wiadomości wymaga zmian w plikach konfiguracyjnych paczki. Ze względu na przyjęte uproszczenie w 
konfiguracji w paczce tsr_materials w pliku **CMakeLists.txt** w miejscu (od 57 linii):

In [ ]:
## Generate services in the 'srv' folder
 add_service_files(
   FILES
   Task.srv
   Nowy.srv
#   Service2.srv
 )

Należy dopisać nazwę swojego serwisu jak powyżej. Zamiast **Nowy.srv** należy podać nazwę serwisu pod jakim został on zapisany w katalogu srv.

In [ ]:
!rossrv show pkg_tsr/... # nazwa utworzonego serwisu, jesli wszystko zostalo prawidlowo utworzone 
# powinna pojawic sie jego struktura